# Stitch Monthly-Median GCS Shards into Annual Rasters + Keep-Mask QA

This notebook assembles the Sentinel-2 monthly-median reference products for **2016–2025**.

Each Earth Engine export is already a 192-band annual stack (15 variables × 12 months + 12 `nObs` bands). The GCS TIFFs are spatial shards, so the notebook mosaics the shards once per year, writes one annual GeoTIFF, then QA/QCs March, July, and October RGB within the keep mask.

**Path policy:** no TIFF is read from or written to a relative path.

- Scratch shards/VRTs: `C:\NCA_DATA\scratch\S2_monthly_medians_stitch`
- Annual mosaics: `C:\NCA_DATA\S2_monthly_medians\annual_mosaics`
- QA: `C:\NCA_DATA\S2_monthly_medians\QAQC`

Edit only `KEEP_MASK_PATH` to point at the current 1=keep / 0=exclude mask.


In [1]:
# Optional one-time install
# %pip install -q numpy pandas matplotlib rasterio google-cloud-storage

from pathlib import Path
import math, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## 1. Configuration

In [2]:
YEARS = list(range(2016, 2026))
QA_MONTHS = [3, 7, 10]

GCS_BUCKET = "bop-nca-data-space-s2-c1-staging"
HISTORICAL_ROOT = "monthly_medians/harmonic_feature_space_2016_2017"
NATIVE_ROOT = "monthly_medians/harmonic_feature_space_2018_2025"

DATA_ROOT = Path(r"C:\NCA_DATA")
SCRATCH_ROOT = DATA_ROOT / "scratch" / "S2_monthly_medians_stitch"
SHARD_ROOT = SCRATCH_ROOT / "gcs_shards"
VRT_ROOT = SCRATCH_ROOT / "vrt"
ANNUAL_ROOT = DATA_ROOT / "S2_monthly_medians" / "annual_mosaics"
QA_ROOT = DATA_ROOT / "S2_monthly_medians" / "QAQC"

# EDIT THIS PATH.
KEEP_MASK_PATH = Path(r"C:\NCA_DATA\templates\Masks\NCA_keepMask_v2.tif")

for p in [DATA_ROOT, SCRATCH_ROOT, SHARD_ROOT, VRT_ROOT,
          ANNUAL_ROOT, QA_ROOT, KEEP_MASK_PATH]:
    if not p.is_absolute():
        raise ValueError(f"Relative path is not allowed: {p}")

for p in [SHARD_ROOT, VRT_ROOT, ANNUAL_ROOT, QA_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

REDOWNLOAD_SHARDS = False
OVERWRITE_ANNUAL = False
DELETE_SHARDS_AFTER_STITCH = False
QA_MAX_SIDE = 1600

ANALYSIS_BANDS = [
    "B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
    "NDVI","NDMI","NDRE","BSI","NBR2",
]
MONTHS = list(range(1, 13))

EXPECTED_BANDS = []
for month in MONTHS:
    mm = f"{month:02d}"
    EXPECTED_BANDS.extend([f"{b}_M{mm}" for b in ANALYSIS_BANDS])
    EXPECTED_BANDS.append(f"nObs_M{mm}")

EXPECTED_COUNT = len(EXPECTED_BANDS)

print("Expected bands/year:", EXPECTED_COUNT)
print("Scratch:", SCRATCH_ROOT)
print("Annual output:", ANNUAL_ROOT)
print("QA output:", QA_ROOT)


Expected bands/year: 192
Scratch: C:\NCA_DATA\scratch\S2_monthly_medians_stitch
Annual output: C:\NCA_DATA\S2_monthly_medians\annual_mosaics
QA output: C:\NCA_DATA\S2_monthly_medians\QAQC


## 2. GDAL + path helpers

In [3]:
for cmd in ["gdalbuildvrt", "gdal_translate"]:
    if shutil.which(cmd) is None:
        raise RuntimeError(f"{cmd} is not available in this environment.")

def root_for_year(year):
    return HISTORICAL_ROOT if year <= 2017 else NATIVE_ROOT

def gcs_prefix_for_year(year):
    return (
        f"{root_for_year(year)}/{year}/"
        f"S2SR_MonthlyMedians_{year}"
    )

def shard_dir_for_year(year):
    p = SHARD_ROOT / str(year)
    if not p.is_absolute():
        raise ValueError(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def vrt_path_for_year(year):
    return VRT_ROOT / f"S2SR_MonthlyMedians_{year}.vrt"

def annual_path_for_year(year):
    return ANNUAL_ROOT / f"S2SR_MonthlyMedians_{year}_annual.tif"

for year in YEARS:
    print(year, "->", f"gs://{GCS_BUCKET}/{gcs_prefix_for_year(year)}")


2016 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2016_2017/2016/S2SR_MonthlyMedians_2016
2017 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2016_2017/2017/S2SR_MonthlyMedians_2017
2018 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2018/S2SR_MonthlyMedians_2018
2019 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2019/S2SR_MonthlyMedians_2019
2020 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2020/S2SR_MonthlyMedians_2020
2021 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2021/S2SR_MonthlyMedians_2021
2022 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2022/S2SR_MonthlyMedians_2022
2023 -> gs://bop-nca-data-space-s2-c1-staging/monthly_medians/harmonic_feature_space_2018_2025/2023/S2SR_MonthlyMedians_2023


## 3. Discover and download GCS shards

In [4]:
from google.cloud import storage
storage_client = storage.Client()

def discover_year_blobs(year):
    return sorted(
        [
            b for b in storage_client.list_blobs(
                GCS_BUCKET,
                prefix=gcs_prefix_for_year(year),
            )
            if b.name.lower().endswith(".tif")
        ],
        key=lambda b: b.name,
    )

def download_year_shards(year):
    blobs = discover_year_blobs(year)
    if not blobs:
        raise FileNotFoundError(
            f"No TIFF shards for {year}: "
            f"gs://{GCS_BUCKET}/{gcs_prefix_for_year(year)}"
        )

    out_dir = shard_dir_for_year(year)
    rows = []

    for blob in blobs:
        out = out_dir / Path(blob.name).name
        if not out.is_absolute():
            raise ValueError(f"Relative TIFF path: {out}")

        remote_size = int(blob.size or 0)
        need = (
            REDOWNLOAD_SHARDS
            or not out.exists()
            or out.stat().st_size != remote_size
        )

        if need:
            print(f"{year}: downloading {out.name}")
            blob.download_to_filename(str(out))

        rows.append({
            "year": year,
            "gcs_object": blob.name,
            "local_path": str(out),
            "remote_bytes": remote_size,
            "local_bytes": out.stat().st_size,
            "size_match": out.stat().st_size == remote_size,
        })

    df = pd.DataFrame(rows)
    if not df["size_match"].all():
        raise RuntimeError(f"{year}: shard byte-size mismatch.")
    return df

manifests = []
for year in YEARS:
    df = download_year_shards(year)
    manifests.append(df)
    print(f"{year}: {len(df)} shard(s)")

download_manifest = pd.concat(manifests, ignore_index=True)
display(download_manifest)
download_manifest.to_csv(QA_ROOT / "GCS_shard_manifest.csv", index=False)


2016: 20 shard(s)
2017: downloading S2SR_MonthlyMedians_20170000005120-0000002560.tif
2017: downloading S2SR_MonthlyMedians_20170000005120-0000005120.tif
2017: downloading S2SR_MonthlyMedians_20170000005120-0000007680.tif
2017: downloading S2SR_MonthlyMedians_20170000005120-0000010240.tif
2017: downloading S2SR_MonthlyMedians_20170000007680-0000000000.tif
2017: downloading S2SR_MonthlyMedians_20170000007680-0000002560.tif
2017: downloading S2SR_MonthlyMedians_20170000007680-0000005120.tif
2017: downloading S2SR_MonthlyMedians_20170000007680-0000007680.tif
2017: downloading S2SR_MonthlyMedians_20170000007680-0000010240.tif
2017: 20 shard(s)
2018: downloading S2SR_MonthlyMedians_20180000000000-0000000000.tif
2018: downloading S2SR_MonthlyMedians_20180000000000-0000002560.tif
2018: downloading S2SR_MonthlyMedians_20180000000000-0000005120.tif
2018: downloading S2SR_MonthlyMedians_20180000000000-0000007680.tif
2018: downloading S2SR_MonthlyMedians_20180000000000-0000010240.tif
2018: downlo

,year,gcs_object,local_path,remote_bytes,local_bytes,size_match
0,2016,monthly_medians/harmonic_feature_space_2016_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,30709713,30709713,True
1,2016,monthly_medians/harmonic_feature_space_2016_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,16430044,16430044,True
2,2016,monthly_medians/harmonic_feature_space_2016_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,15710066,15710066,True
3,2016,monthly_medians/harmonic_feature_space_2016_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,14899552,14899552,True
4,2016,monthly_medians/harmonic_feature_space_2016_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,1882532,1882532,True
...,...,...,...,...,...,...
195,2025,monthly_medians/harmonic_feature_space_2018_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,1166892,1166892,True
196,2025,monthly_medians/harmonic_feature_space_2018_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,1166892,1166892,True
197,2025,monthly_medians/harmonic_feature_space_2018_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,1166892,1166892,True
198,2025,monthly_medians/harmonic_feature_space_2018_20...,C:\NCA_DATA\scratch\S2_monthly_medians_stitch\...,1295770,1295770,True


## 4. Validate shard schema

In [5]:
def local_year_shards(year):
    paths = sorted(shard_dir_for_year(year).glob("*.tif"))
    for p in paths:
        if not p.is_absolute():
            raise ValueError(f"Relative TIFF path: {p}")
    return paths

def meta(path):
    with rasterio.open(path) as src:
        return {
            "file": path.name,
            "count": src.count,
            "dtype": ",".join(sorted(set(src.dtypes))),
            "crs": str(src.crs),
            "res_x": abs(src.transform.a),
            "res_y": abs(src.transform.e),
            "nodata": src.nodata,
            "descriptions": list(src.descriptions),
        }

rows = []
for year in YEARS:
    paths = local_year_shards(year)
    if not paths:
        raise FileNotFoundError(f"No local shards for {year}")

    metas = [meta(p) for p in paths]
    first = metas[0]

    if first["count"] != EXPECTED_COUNT:
        raise RuntimeError(
            f"{year}: expected {EXPECTED_COUNT} bands, "
            f"found {first['count']}."
        )

    for m in metas[1:]:
        for key in ["count","dtype","crs","res_x","res_y","nodata"]:
            if m[key] != first[key]:
                raise RuntimeError(
                    f"{year}: shard mismatch for {key}: {m['file']}"
                )

    desc = first["descriptions"]
    desc_present = all(x is not None for x in desc)
    missing = [b for b in EXPECTED_BANDS if b not in desc] if desc_present else []
    extra = [b for b in desc if b not in EXPECTED_BANDS] if desc_present else []

    rows.append({
        "year": year,
        "n_shards": len(paths),
        "band_count": first["count"],
        "dtype": first["dtype"],
        "crs": first["crs"],
        "res_x": first["res_x"],
        "res_y": first["res_y"],
        "nodata": first["nodata"],
        "descriptions_present": desc_present,
        "missing_expected_bands": len(missing),
        "unexpected_bands": len(extra),
    })

shard_schema = pd.DataFrame(rows)
display(shard_schema)
shard_schema.to_csv(QA_ROOT / "annual_shard_schema.csv", index=False)


,year,n_shards,band_count,dtype,crs,res_x,res_y,nodata,descriptions_present,missing_expected_bands,unexpected_bands
0,2016,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
1,2017,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
2,2018,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
3,2019,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
4,2020,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
5,2021,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
6,2022,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
7,2023,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
8,2024,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0
9,2025,20,192,float32,"PROJCS[""NAD83 / UTM zone 11N"",GEOGCS[""NAD83"",D...",10.0,10.0,-32768.0,True,0,0


## 5. Stitch spatial shards into one annual 192-band GeoTIFF

Because every shard contains all monthly bands, mosaicking once at the annual level stitches March, July, October, and every other month simultaneously.


In [ ]:
def build_annual_mosaic(year):
    shards = local_year_shards(year)
    vrt = vrt_path_for_year(year)
    annual = annual_path_for_year(year)

    if annual.exists() and not OVERWRITE_ANNUAL:
        print(f"{year}: reusing existing annual TIFF")
        return annual

    subprocess.run(
        ["gdalbuildvrt", "-overwrite", str(vrt)]
        + [str(p) for p in shards],
        check=True,
    )

    with rasterio.open(shards[0]) as src:
        is_float = np.dtype(src.dtypes[0]).kind == "f"

    predictor = "3" if is_float else "2"

    subprocess.run(
        [
            "gdal_translate",
            "-of", "GTiff",
            "-co", "BIGTIFF=YES",
            "-co", "TILED=YES",
            "-co", "COMPRESS=DEFLATE",
            "-co", f"PREDICTOR={predictor}",
            "-co", "NUM_THREADS=ALL_CPUS",
            str(vrt),
            str(annual),
        ],
        check=True,
    )

    with rasterio.open(annual, "r+") as dst:
        if dst.count != EXPECTED_COUNT:
            raise RuntimeError(
                f"{year}: stitched output has {dst.count} bands."
            )
        if not all(x is not None for x in dst.descriptions):
            for i, name in enumerate(EXPECTED_BANDS, start=1):
                dst.set_band_description(i, name)

    print(f"{year}: wrote {annual}")
    return annual

annual_paths = {}
for year in YEARS:
    annual_paths[year] = build_annual_mosaic(year)


## 6. Validate annual mosaics

In [ ]:
rows = []

for year, path in annual_paths.items():
    with rasterio.open(path) as src:
        desc = list(src.descriptions)
        rows.append({
            "year": year,
            "path": str(path),
            "size_GiB": path.stat().st_size / 1024**3,
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "crs": str(src.crs),
            "res_x": abs(src.transform.a),
            "res_y": abs(src.transform.e),
            "nodata": src.nodata,
            "schema_matches_expected": (
                src.count == EXPECTED_COUNT
                and desc == EXPECTED_BANDS
            ),
        })

annual_inventory = pd.DataFrame(rows)
display(annual_inventory)
annual_inventory.to_csv(
    QA_ROOT / "annual_mosaic_inventory.csv",
    index=False,
)


## 7. Keep-mask RGB QA

For each year, display true-color **March, July, and October** using:

- Red = B4
- Green = B3
- Blue = B2

The keep mask is warped with nearest-neighbor resampling to the annual raster grid. The 2–98% display stretch is calculated only from retained pixels. The notebook also records the valid RGB fraction and median monthly `nObs` within the keep mask.


In [ ]:
if not KEEP_MASK_PATH.exists():
    raise FileNotFoundError(
        f"Update KEEP_MASK_PATH before QA: {KEEP_MASK_PATH}"
    )

def band_index(src, name):
    desc = list(src.descriptions)
    if name not in desc:
        raise KeyError(f"Band not found: {name}")
    return desc.index(name) + 1

def read_month_preview(annual_path, month, max_side=QA_MAX_SIDE):
    mm = f"{month:02d}"

    with rasterio.open(annual_path) as src:
        scale = max(src.width, src.height) / max_side
        scale = max(scale, 1)
        out_w = max(1, int(src.width / scale))
        out_h = max(1, int(src.height / scale))

        rgb_idx = [
            band_index(src, f"B4_M{mm}"),
            band_index(src, f"B3_M{mm}"),
            band_index(src, f"B2_M{mm}"),
        ]
        nobs_idx = band_index(src, f"nObs_M{mm}")

        rgb = src.read(
            rgb_idx,
            out_shape=(3, out_h, out_w),
            resampling=Resampling.average,
        ).astype("float32")

        nobs = src.read(
            nobs_idx,
            out_shape=(out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        if src.nodata is not None:
            rgb[rgb == src.nodata] = np.nan
            nobs[nobs == src.nodata] = np.nan

        with rasterio.open(KEEP_MASK_PATH) as mask_src:
            with WarpedVRT(
                mask_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=Resampling.nearest,
            ) as mask_vrt:
                keep = mask_vrt.read(
                    1,
                    out_shape=(out_h, out_w),
                    resampling=Resampling.nearest,
                )

    keep = np.isfinite(keep) & (keep > 0)
    valid_rgb = keep & np.all(np.isfinite(rgb), axis=0)

    display_rgb = np.zeros((out_h, out_w, 3), dtype="float32")

    for channel in range(3):
        vals = rgb[channel][valid_rgb]
        if vals.size == 0:
            continue
        lo, hi = np.nanpercentile(vals, [2, 98])
        if hi <= lo:
            continue
        stretch = np.clip((rgb[channel] - lo) / (hi - lo), 0, 1)
        display_rgb[:, :, channel] = np.where(valid_rgb, stretch, 0)

    keep_n = int(keep.sum())
    valid_n = int(valid_rgb.sum())
    nobs_ok = keep & np.isfinite(nobs)

    qa = {
        "keep_preview_pixels": keep_n,
        "valid_rgb_preview_pixels": valid_n,
        "valid_rgb_fraction_within_keep": (
            valid_n / keep_n if keep_n else np.nan
        ),
        "median_nObs_within_keep": (
            float(np.nanmedian(nobs[nobs_ok]))
            if nobs_ok.any() else np.nan
        ),
    }

    return display_rgb, qa


## 8. Run March / July / October QA for every year

In [ ]:
qa_rows = []

for year in YEARS:
    print("\n" + "=" * 70)
    print(year)
    print("=" * 70)

    for month in QA_MONTHS:
        rgb, qa = read_month_preview(
            annual_paths[year],
            month,
        )

        qa_rows.append({
            "year": year,
            "month": month,
            **qa,
        })

        fig_path = (
            QA_ROOT
            / f"S2SR_MonthlyMedians_{year}_M{month:02d}_RGB.png"
        )

        plt.figure(figsize=(10, 8))
        plt.imshow(rgb)
        plt.title(
            f"{year} — Month {month:02d} — RGB within keep mask"
        )
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(fig_path, dpi=160, bbox_inches="tight")
        plt.show()

        print(
            f"M{month:02d}: valid RGB fraction = "
            f"{qa['valid_rgb_fraction_within_keep']:.3f}; "
            f"median nObs = {qa['median_nObs_within_keep']:.1f}"
        )

qa_table = pd.DataFrame(qa_rows)
display(qa_table)

qa_table.to_csv(
    QA_ROOT / "March_July_October_RGB_QA.csv",
    index=False,
)


## 9. Optional scratch cleanup

In [ ]:
if DELETE_SHARDS_AFTER_STITCH:
    for year in YEARS:
        annual = annual_path_for_year(year)
        if not annual.exists():
            raise RuntimeError(
                f"Refusing cleanup: annual TIFF missing for {year}"
            )
        shard_dir = shard_dir_for_year(year)
        print("Deleting:", shard_dir)
        shutil.rmtree(shard_dir)
else:
    print(
        "DELETE_SHARDS_AFTER_STITCH = False; "
        "scratch shards retained."
    )


## Outputs

Annual mosaics:

```text
C:\NCA_DATA\S2_monthly_medians\annual_mosaics\
    S2SR_MonthlyMedians_2016_annual.tif
    ...
    S2SR_MonthlyMedians_2025_annual.tif
```

QA:

```text
C:\NCA_DATA\S2_monthly_medians\QAQC\
    GCS_shard_manifest.csv
    annual_shard_schema.csv
    annual_mosaic_inventory.csv
    March_July_October_RGB_QA.csv
    S2SR_MonthlyMedians_<YEAR>_M03_RGB.png
    S2SR_MonthlyMedians_<YEAR>_M07_RGB.png
    S2SR_MonthlyMedians_<YEAR>_M10_RGB.png
```

The RGB previews are intended to catch seam artifacts, missing shards, incorrect band order, anomalous masking, or obvious historical/native inconsistencies without an expensive full-resolution census.
